# Multi-Agent Company & Finance Research Assistant — Demo

This notebook reproduces the project end-to-end:
1. Configure the environment (Google Gemini API key)
2. Inspect the **Data-Analyst** tools (no LLM needed)
3. Build the **knowledge base** for the Document-Q&A (RAG) agent
4. Run each specialist and the full **supervisor** multi-agent system
5. Run a slice of the **evaluation harness**

> Architecture: a LangGraph *supervisor* routes each query to one of three
> specialists (`document_qa`, `data_analyst`, `web_research`), can chain them,
> and a *synthesizer* composes the final answer.

## 0. Setup
Run from the **project root**. Install dependencies and set your key.
Get a free key at https://aistudio.google.com/apikey

In [ ]:
# !pip install -r ../requirements.txt   # uncomment on first run
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # make 'src' and 'eval' importable

# Set your key here, or rely on a .env file in the project root.
# os.environ['GOOGLE_API_KEY'] = 'your_key_here'
from dotenv import load_dotenv
load_dotenv('../.env')
print('GOOGLE_API_KEY set:', bool(os.environ.get('GOOGLE_API_KEY')))

## 1. Data-Analyst tools (deterministic, no LLM)
These pandas-backed tools give the analyst agent reliable, verifiable numbers.

In [ ]:
from src import tools
print(tools.dataset_overview.invoke({}))
print()
print(tools.top_companies.invoke({'metric': 'market_cap_usd_b', 'n': 5}))
print()
print(tools.aggregate.invoke({'group_by': 'sector', 'metric': 'revenue_usd_b', 'agg': 'sum'}))

## 2. Build the knowledge base (RAG)
First run scrapes Wikipedia and embeds the corpus into Chroma (slow, one-time);
later runs load the persisted store from `.chroma/`.

In [ ]:
from src.knowledge_base import get_retriever
retriever = get_retriever()
docs = retriever.invoke('What is a price to earnings ratio?')
for d in docs:
    print('-', d.metadata['title'])

## 3. Run the full multi-agent system
Watch the supervisor route each query. The first two stay within one
specialist; the last is a *multi-hop* query needing two specialists.

In [ ]:
from src.agents import ask

print(ask('What is market capitalization?', verbose=True))

In [ ]:
print(ask('Which sector has the highest total revenue in the dataset?', verbose=True))

In [ ]:
# Multi-hop: define a concept (document_qa) THEN rank the dataset (data_analyst)
print(ask('Explain what market capitalization means, then list the top 3 companies by it in the dataset.', verbose=True))

## 4. Evaluation (slice)
Run the quantitative harness. Full run: `python -m eval.run_eval --all`

In [ ]:
from eval.run_eval import load_items, eval_routing, eval_analyst
items = load_items()
routing = eval_routing(items)
analyst = eval_analyst(items)
print('\nRouting accuracy:', routing['accuracy'])
print('Analyst correctness:', analyst['accuracy'])